In [1]:
!pip install joblib
!pip install requests

In [2]:
import pandas as pd
import requests
import numpy as np

In [3]:
# ==============================================================================
# 1. Configuration & API Data Fetching
# ==============================================================================
# Location parameters for Lund, Sweden
LATITUDE = 55.71
LONGITUDE = 13.19
START_DATE = "2010-01-01"
END_DATE = "2024-12-31"

# Open-Meteo Historical Archive API Endpoint
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "dew_point_2m",
        "precipitation",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
    ],
    "timezone": "Europe/Berlin",
}

print("Fetching historical weather observations for Lund from Open-Meteo...")
response = requests.get(url, params=params, timeout=20)
response.raise_for_status()
data = response.json()

Fetching historical weather observations for Lund from Open-Meteo...


In [4]:
# ==============================================================================
# 2. Raw Hourly Data Structuring
# ==============================================================================
df_hourly = pd.DataFrame(data["hourly"])
df_hourly["time"] = pd.to_datetime(df_hourly["time"])
df_hourly["date"] = df_hourly["time"].dt.date

In [5]:
# ==============================================================================
# 3. Daily Static Feature Aggregation (Features for Day n)
# ==============================================================================
print("Aggregating hourly measurements into daily feature representations...")
df_daily = (
    df_hourly.groupby("date")
    .agg(
        temp_max=("temperature_2m", "max"),
        temp_min=("temperature_2m", "min"),
        temp_mean=("temperature_2m", "mean"),
        humidity_mean=("relative_humidity_2m", "mean"),
        dewpoint_mean=("dew_point_2m", "mean"),
        pressure_mean=("surface_pressure", "mean"),
        cloud_cover_mean=("cloud_cover", "mean"),
        wind_speed_max=("wind_speed_10m", "max"),
        precipitation_today=("precipitation", "sum"),
    )
    .reset_index()
)

Aggregating hourly measurements into daily feature representations...


In [6]:
# ==============================================================================
# 4. Feature Engineering
# ==============================================================================
# Diurnal temperature range (Day n)
df_daily["temp_range"] = df_daily["temp_max"] - df_daily["temp_min"]

# Dewpoint spread: Proxy for atmospheric moisture saturation (Day n)
df_daily["dewpoint_spread"] = df_daily["temp_mean"] - df_daily["dewpoint_mean"]

# Day-over-day barometric pressure trend
df_daily["pressure_change"] = df_daily["pressure_mean"].diff().fillna(0.0)


df_daily["precipitation_tomorrow"] = df_daily["precipitation_today"].shift(-1)

df_daily["rain_tomorrow"] = (df_daily["precipitation_tomorrow"] > 0.1).astype(int)

df_daily = df_daily.iloc[:-1].reset_index(drop=True)

# Seasonal cycle signal
month_series = pd.to_datetime(df_daily["date"]).dt.month
df_daily["month_sin"] = np.sin(2 * np.pi * month_series / 12.0)
df_daily["month_cos"] = np.cos(2 * np.pi * month_series / 12.0)

In [7]:
# ==============================================================================
# 5. Target Label Alignment (Label for Day n + 1)
# ==============================================================================
# Shift target labels backward using shift(-1) so that meteorological features
# from Day n are paired with the binary precipitation outcome of Day n + 1.
# RainTomorrow = 1 if Day n+1 precipitation > 0.1 mm, otherwise 0.
df_daily["rain_tomorrow"] = (
    df_daily["precipitation_today"].shift(-1) > 0.1
).astype(int)

# Drop the final row as shift(-1) creates a NaN label for the last day
df_daily = df_daily.iloc[:-1].reset_index(drop=True)

In [8]:
# ==============================================================================
# 6. Save Processed Dataset to Local Storage
# ==============================================================================
output_csv = "lund_daily_weather.csv"
df_daily.to_csv(output_csv, index=False)

print(
    f"\nDataset processing complete! Created {len(df_daily)} daily samples."
)
print(f"Saved dataset to '{output_csv}'.")
print("\nFirst 5 rows of the aligned tabular dataset:")
print(df_daily.head())


Dataset processing complete! Created 5477 daily samples.
Saved dataset to 'lund_daily_weather.csv'.

First 5 rows of the aligned tabular dataset:
         date  temp_max  temp_min  temp_mean  humidity_mean  dewpoint_mean  \
0  2010-01-01      -4.7      -8.3  -6.741667      80.208333      -9.629167   
1  2010-01-02      -4.0      -9.5  -6.683333      78.291667      -9.900000   
2  2010-01-03      -5.9     -11.5  -9.350000      75.791667     -12.862500   
3  2010-01-04      -1.0      -6.1  -3.137500      86.541667      -5.079167   
4  2010-01-05      -0.7      -2.6  -1.825000      93.166667      -2.779167   

   pressure_mean  cloud_cover_mean  wind_speed_max  precipitation_today  \
0     996.700000         83.125000            14.1                  0.0   
1    1007.425000         22.875000            19.4                  0.0   
2    1013.662500         33.250000            11.3                  0.0   
3    1004.058333         85.000000            24.3                  1.4   
4     999